# Insurance Premium Prediction — Google Colab Pipeline
## CPE232 Data Models - Hackathon 2

## Overview
This notebook is designed for Google Colab to participate in the Insurance Premium Prediction Hackathon. 
It includes:
- **Mounting Google Drive** for data access.
- **Data Preprocessing** (Feature Engineering, Missing Value Imputation).
- **Ensemble Modeling** (XGBoost + LightGBM + CatBoost).
- **GPU Acceleration** enabled for faster training.

## Dataset
- **Metric:** Mean Absolute Error (MAE).
- **Target:** `Premium Amount`.


## 1. Environment Setup & Import
Install necessary libraries if not present and import dependencies.

In [ ]:
!pip install catboost xgboost lightgbm --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Mount Google Drive & Load Data
Ensure your data is uploaded to your Google Drive.

In [ ]:
from google.colab import drive
import os

# Mount Drive
try:
    drive.mount('/content/drive')
    print("Google Drive mounted.")
except:
    print("Drive mount failed or running locally.")

# Configuration - Adjust this path to your folder in Drive
# Example: "/content/drive/MyDrive/Hackathon2/cpe-232-insurance-premium-prediction"
BASE_PATH = "/content/drive/MyDrive/cpe232-datamodel-2025/hackathon/Hackathon2_Insurance-Premium-Prediction/cpe-232-insurance-premium-prediction"

train_path  = f"{BASE_PATH}/train.csv"
test_path   = f"{BASE_PATH}/test.csv"
sample_path = f"{BASE_PATH}/sample_submission.csv"

# Fallback to local upload if drive path doesn't exist
if not os.path.exists(train_path):
    print(f"Path not found: {BASE_PATH}")
    print("Please manually upload files to /content/ or fix the path.")
    BASE_PATH = "/content"
    train_path  = f"{BASE_PATH}/train.csv"
    test_path   = f"{BASE_PATH}/test.csv"
    sample_path = f"{BASE_PATH}/sample_submission.csv"

print(f"Loading data from: {train_path}")

try:
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    sample_sub = pd.read_csv(sample_path)
    print(f"Train Shape: {train.shape}")
    print(f"Test Shape:  {test.shape}")
except FileNotFoundError:
    print("files not found! Please upload train.csv, test.csv, sample_submission.csv")

## 3. Exploratory Data Analysis (EDA)
Quick check on target distribution and correlations.

In [ ]:
# Target Distribution
plt.figure(figsize=(10, 5))
sns.histplot(train['Premium Amount'], bins=50, kde=True, color='green')
plt.title('Distribution of Premium Amount')
plt.show()

# Correlation Matrix
num_cols = train.select_dtypes(include=[np.number]).columns.drop(['id', 'Premium Amount'], errors='ignore')
corr = train[num_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

## 4. Preprocessing & Feature Engineering
- Parse Dates.
- Impute Missing Values.
- Encode Categoricals.

In [ ]:
# Clean column names (remove leading/trailing spaces)
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

def preprocess_date(df):
    date_col = 'Policy Start Date'
    
    # Check if the column exists
    if date_col not in df.columns:
        print(f"Warning: '{date_col}' not found. Searching for alternatives...")
        found = False
        for col in df.columns:
            if 'start date' in col.lower() or 'policy date' in col.lower():
                print(f"Found alternative date column: '{col}'")
                date_col = col
                found = True
                break
        
        if not found:
            print(f"Error: Could not find date column. Available columns: {list(df.columns)}")
            return df
    
    # Keep original column name for consistency or rename? Let's use the found name but process it.
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    
    # Feature Engineering from Date
    df['Policy_Year'] = df[date_col].dt.year
    df['Policy_Month'] = df[date_col].dt.month
    df['Policy_Day'] = df[date_col].dt.day
    df['Policy_DayOfWeek'] = df[date_col].dt.dayofweek
    
    # Calculate Policy Duration in Days (relative to the latest date in dataset)
    ref_date = df[date_col].max()
    df['Policy_Age_Days'] = (ref_date - df[date_col]).dt.days
    
    return df

def feature_engineering(df):
    if 'Customer Feedback' in df.columns:
        # Text length feature
        df['Feedback_Len'] = df['Customer Feedback'].astype(str).apply(len)
    else:
         print("Warning: 'Customer Feedback' col not found, skipping feature.")
    return df

print("Processing Features...")
train = preprocess_date(train)
test = preprocess_date(test)

train = feature_engineering(train)
test = feature_engineering(test)

# Drop Columns that won't be used for training
# Note: Need to drop the actual date column found above if it wasn't renamed.
# For safety, let's drop any column with 'Date' in it that we processed, or specific list.
drop_cols = ['id', 'Customer Feedback', 'Policy Start Date']
# Ensure columns exist before dropping
cols_to_drop = [c for c in drop_cols if c in train.columns]

X = train.drop(columns=['Premium Amount'] + cols_to_drop, errors='ignore')
y = train['Premium Amount']
X_test = test.drop(columns=cols_to_drop, errors='ignore')

# Imputation
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical Features: {len(num_features)}")
print(f"Categorical Features: {len(cat_features)}")

# Numerical -> Median
if len(num_features) > 0:
    imputer_num = SimpleImputer(strategy='median')
    X[num_features] = imputer_num.fit_transform(X[num_features])
    X_test[num_features] = imputer_num.transform(X_test[num_features])

# Categorical -> Most Frequent
if len(cat_features) > 0:
    imputer_cat = SimpleImputer(strategy='most_frequent')
    X[cat_features] = imputer_cat.fit_transform(X[cat_features])
    X_test[cat_features] = imputer_cat.transform(X_test[cat_features])

# Encoding Categorical Variables
# Label Encoding is effective for Tree-based models (XGB/LGB/CatBoost)
for col in cat_features:
    le = LabelEncoder()
    # Fit on both train and test to cover all categories
    full_data = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(full_data)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

print("Preprocessing Complete.")

## 5. GPU-Accelerated Model Training
We use XGBoost and CatBoost with GPU support.

In [ ]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Global placeholders for storage
oof_preds_xgb = np.zeros(len(X))
test_preds_xgb = np.zeros(len(X_test))

oof_preds_lgb = np.zeros(len(X))
test_preds_lgb = np.zeros(len(X_test))

oof_preds_cat = np.zeros(len(X))
test_preds_cat = np.zeros(len(X_test))

# --- 1. XGBoost ---
print("\n========== Training XGBoost (GPU) ==========")
xgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.05,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'reg:absoluteerror',
    'n_jobs': -1,
    'random_state': 42,
    'eval_metric': 'mae',
    'early_stopping_rounds': 100,
    # GPU Configuration for XGBoost 2.0+
    'tree_method': 'hist', 
    'device': 'cuda' 
}

# Optional: Disable GPU if not available (Colab Safe Check)
try:
    import torch
    if not torch.cuda.is_available():
        xgb_params['device'] = 'cpu'
        print("XGBoost: GPU not found, using CPU.")
except:
        pass

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    oof_preds_xgb[val_idx] = model.predict(X_val)
    test_preds_xgb += model.predict(X_test) / N_FOLDS
    print(f"Fold {fold+1} MAE: {mean_absolute_error(y_val, oof_preds_xgb[val_idx]):.4f}")

mae_xgb = mean_absolute_error(y, oof_preds_xgb)
print(f"XGBoost Overall MAE: {mae_xgb:.4f}")


# --- 2. LightGBM ---
print("\n========== Training LightGBM ==========")
lgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'objective': 'mae',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**lgb_params)
    
    callbacks = [lgb.early_stopping(stopping_rounds=100, verbose=False)]
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='mae', callbacks=callbacks)
    
    oof_preds_lgb[val_idx] = model.predict(X_val)
    test_preds_lgb += model.predict(X_test) / N_FOLDS
    print(f"Fold {fold+1} MAE: {mean_absolute_error(y_val, oof_preds_lgb[val_idx]):.4f}")

mae_lgb = mean_absolute_error(y, oof_preds_lgb)
print(f"LightGBM Overall MAE: {mae_lgb:.4f}")


# --- 3. CatBoost ---
print("\n========== Training CatBoost (GPU) ==========")
cat_params = {
    'iterations': 2000,
    'learning_rate': 0.05,
    'depth': 8,
    'loss_function': 'MAE',
    'verbose': 0,
    'random_state': 42,
    'task_type': 'GPU', # Enable GPU
    'devices': '0'
}

# Optional: Disable GPU if not available (Colab Safe Check)
try:
    import torch
    if not torch.cuda.is_available():
        del cat_params['task_type']
        del cat_params['devices']
        print("CatBoost: GPU not found, using CPU.")
except:
    pass

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = CatBoostRegressor(**cat_params)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)
    
    oof_preds_cat[val_idx] = model.predict(X_val)
    test_preds_cat += model.predict(X_test) / N_FOLDS
    print(f"Fold {fold+1} MAE: {mean_absolute_error(y_val, oof_preds_cat[val_idx]):.4f}")

mae_cat = mean_absolute_error(y, oof_preds_cat)
print(f"CatBoost Overall MAE: {mae_cat:.4f}")

## 6. Submission & Ensemble
Weighted average ensemble.

In [ ]:
w_xgb = 0.5
w_cat = 0.5

# Simple Ensemble
final_preds = (w_xgb * test_preds_xgb) + (w_cat * test_preds_cat)

submission = pd.DataFrame({
    'id': sample_sub['id'],
    'Premium Amount': final_preds
})
submission.to_csv('submission.csv', index=False)
print("Saved submission.csv")
submission.head()